### Data Ingestion

#### Data Structure

In [1]:
from langchain_core.documents import Document

In [ ]:
### PDF Loader - Single

from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/text_files/PythonBasics.pdf", mode="page")

document = loader.load()

print(document[50])

page_content='Panchatcharam M
December 2025
Boolean Datatypes
>>> is_pass = True
 >>> type (is_pass)
<class ‘bool’>
51
>>> is_pass = False
 >>> type (is_pass)
<class ‘bool’>' metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2025-12-04T08:59:38+05:30', 'source': '../data/text_files/PythonBasics.pdf', 'file_path': '../data/text_files/PythonBasics.pdf', 'total_pages': 193, 'format': 'PDF 1.7', 'title': 'PowerPoint Presentation', 'author': 'Panchatcharam Mariappan', 'subject': '', 'keywords': '', 'moddate': '2025-12-04T08:59:38+05:30', 'trapped': '', 'modDate': "D:20251204085938+05'30'", 'creationDate': "D:20251204085938+05'30'", 'page': 50}


In [18]:
### PDF Loader - Multiple files

from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

# Load all PDF files in the directory
loader = DirectoryLoader(
    "../data/text_files/", 
    glob="**/*.pdf", 
    show_progress=False, 
    loader_cls=PyMuPDFLoader)

documents = loader.load()

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 534


In [34]:
### Document Chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\nFirst chunk preview:")
print(f"Content: {chunks[0].page_content[:200]}...")
print(f"Metadata: {chunks[0].metadata}")

Total chunks created: 931

First chunk preview:
Content: CS551
1
CS551: Introduction to Deep Learning
Arijit Mondal
Dept. of Computer Science & Engineering
Indian Institute of Technology Patna
arijit@iitp.ac.in...
Metadata: {'producer': 'xdvipdfmx (20190824)', 'creator': 'LaTeX with Beamer class', 'creationdate': '2024-01-03T15:44:28+05:30', 'source': '..\\data\\text_files\\deep_learning.pdf', 'file_path': '..\\data\\text_files\\deep_learning.pdf', 'total_pages': 57, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20240103154428+05'30'", 'page': 0}


In [35]:
chunks

[Document(metadata={'producer': 'xdvipdfmx (20190824)', 'creator': 'LaTeX with Beamer class', 'creationdate': '2024-01-03T15:44:28+05:30', 'source': '..\\data\\text_files\\deep_learning.pdf', 'file_path': '..\\data\\text_files\\deep_learning.pdf', 'total_pages': 57, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20240103154428+05'30'", 'page': 0}, page_content='CS551\n1\nCS551: Introduction to Deep Learning\nArijit Mondal\nDept. of Computer Science & Engineering\nIndian Institute of Technology Patna\narijit@iitp.ac.in'),
 Document(metadata={'producer': 'xdvipdfmx (20190824)', 'creator': 'LaTeX with Beamer class', 'creationdate': '2024-01-03T15:44:28+05:30', 'source': '..\\data\\text_files\\deep_learning.pdf', 'file_path': '..\\data\\text_files\\deep_learning.pdf', 'total_pages': 57, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', '

### Embeddings and Vector Store DB

In [31]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os



In [25]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name =model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loaded model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedded dimension:", self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded.")
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print("Embeddings generated successfully. Shape:", np.array(embeddings).shape)
            return np.array(embeddings)
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
            
### Initializing the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loaded model: all-MiniLM-L6-v2


c:\Users\vijay\Downloads\RAG\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vijay\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1570.86it/s]


Model loaded successfully. Embedded dimension: 384


C:\Users\vijay\AppData\Local\Temp\ipykernel_11908\3891531744.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded successfully. Embedded dimension:", self.model.get_sentence_embedding_dimension())


In [38]:
### Vector Store - ChromaDB

class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            if self.collection_name in self.client.list_collections():
                self.collection = self.client.get_collection(
                    name=self.collection_name,
                    metadata={"description":"PDF document embeddings for RAG system"}
                    )
                print(f"Vector store initialized with collection '{self.collection_name}'.")
                print(f"Collection '{self.collection_name}' loaded successfully.")
            else:
                self.collection = self.client.create_collection(name=self.collection_name)
                print(f"Vector store initialized with collection '{self.collection_name}'.")
                print(f"Collection '{self.collection_name}' created successfully.")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        try:
            if len(documents) != len(embeddings):
                raise ValueError("Number of documents and embeddings must match.")
            print(f"Adding {len(documents)} documents to vector store...")

            # Prepare data for ChromaDB
            ids=[]
            metadatas=[]
            document_texts=[]
            embedding_list=[]

            for i, (doc, emb) in enumerate(zip(documents, embeddings)):
                doc_id = str(uuid.uuid4())
                ids.append(doc_id)

                # prepare metadata
                metadata=dict(doc.metadata) if doc.metadata else {}
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                #Document content
                document_texts.append(doc.page_content)

                # Embeddings
                embedding_list.append(emb.tolist())
            
            # Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embedding_list,
                    metadatas=metadatas,
                    documents=document_texts
                )
                print(f"Successfully added {len(documents)} documents to collection '{self.collection_name}'.")
                print(f"Total documents in collection '{self.collection_name}': {self.collection.count()}.")
            except Exception as e:
                print(f"Error adding documents to collection: {e}")
                raise
        except Exception as e:
            print(f"Error in add_documents: {e}")
            raise

### Initializing the vector store

vector_store = VectorStore()
vector_store




               

Vector store initialized with collection 'pdf_documents'.
Collection 'pdf_documents' created successfully.


In [39]:
### Convert the text chunks into embeddings and add to vector store

texts = [doc.page_content for doc in chunks]

### Generate the embeddings for the text chunks

embeddings = embedding_manager.generate_embeddings(texts)

### Add the documents and their embeddings to the vector store

vector_store.add_documents(chunks, embeddings)



Generating embeddings for 931 texts...


Batches: 100%|██████████| 30/30 [01:04<00:00,  2.15s/it]


Embeddings generated successfully. Shape: (931, 384)
Adding 931 documents to vector store...
Successfully added 931 documents to collection 'pdf_documents'.
Total documents in collection 'pdf_documents': 931.


## Retrival Pipeline from Vector Store

In [ ]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Tuple[Document, float]]:
        try:
            print(f"Retrieving relevant documents for query: '{query}'")
            print(f"Top K: {top_k}, Score threshold: {score_threshold}")

            # Generate embedding for the query
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]

            try:
                results = self.vector_store.collection.query(
                    query_embeddings=[query_embedding.tolist()],
                    n_results=top_k,
                    include=["documents", "metadatas", "embeddings"]
                )
                retrieved_docs = []
                for doc_text, metadata, emb in zip(
                    results['documents'][0], 
                    results['metadatas'][0], 
                    results['embeddings'][0]
                    ):
                    similarity = cosine_similarity([query_embedding], [emb])[0][0]
                    retrieved_docs.append((Document(page_content=doc_text, metadata=metadata), similarity))
                retrieved_docs.sort(key=lambda x: x[1], reverse=True)
                print(f"Retrieved {len(retrieved_docs)} documents.")
                return retrieved_docs
            except Exception as e:
                print(f"Error during query execution: {e}")
                raise
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise

rag_retriever = RAGRetriever(vector_store, embedding_manager)




In [47]:
rag_retriever.retrieve("Applications: Computer vision", top_k=1)

Retrieving relevant documents for query: 'Applications: Computer vision'
Top K: 1, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.04it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 1 documents.


[(Document(metadata={'page': 10, 'doc_index': 10, 'moddate': '', 'author': '', 'modDate': '', 'creationdate': '2024-01-03T15:44:28+05:30', 'total_pages': 57, 'trapped': '', 'creationDate': "D:20240103154428+05'30'", 'producer': 'xdvipdfmx (20190824)', 'source': '..\\data\\text_files\\deep_learning.pdf', 'title': '', 'format': 'PDF 1.5', 'subject': '', 'file_path': '..\\data\\text_files\\deep_learning.pdf', 'keywords': '', 'creator': 'LaTeX with Beamer class', 'content_length': 152}, page_content='Image source: Internet\nCS551\n9\nApplications: Computer vision\n• 2d to 3d conversion\n• Street view generation\n• Image classifications\n• Image segmentation'),
  np.float64(0.5142949524767914))]